## 1. Download and Import libraries

In [ ]:
!pip install langchain-huggingface langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is 

In [ ]:
#For Langgraph
from langchain_core.messages import BaseMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from typing import Annotated, Sequence, TypedDict
#For LLM
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

## 1.1 Class helper

In [ ]:
import re
from typing import List, Dict


class CVTextCleaner:
    def __init__(self):
        # Common OCR / PDF artifacts
        self.artifact_patterns = [
            r'==>.*?<==',
            r'----- Start of picture text -----.*?----- End of picture text -----',
            r'\bPage \d+ of \d+\b',
        ]

        # Section headers (expandable)
        self.section_keywords = [
            "OBJECTIVE",
            "SUMMARY",
            "EXECUTIVE SUMMARY",
            "SKILLS",
            "AREAS OF EXPERTISE",
            "EXPERIENCE",
            "EDUCATION",
            "PROJECTS",
            "PRODUCT PORTFOLIO"
        ]

    # ----------------------------
    # Step 1: Remove artifacts
    # ----------------------------
    def remove_artifacts(self, text: str) -> str:
        for pattern in self.artifact_patterns:
            text = re.sub(pattern, ' ', text, flags=re.DOTALL)
        return text

    # ----------------------------
    # Step 2: Normalize whitespace
    # ----------------------------
    def normalize_whitespace(self, text: str) -> str:
        text = text.replace('\xa0', ' ')
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    # ----------------------------
    # Step 3: Clean special characters
    # ----------------------------
    def clean_symbols(self, text: str) -> str:
        text = re.sub(r'[©«»•]', '', text)
        text = re.sub(r'[\*\|]', ' ', text)
        return text

    # ----------------------------
    # Step 4: Normalize bullets
    # ----------------------------
    def normalize_bullets(self, text: str) -> str:
        text = re.sub(r'\s*[-•*]\s*', '\n- ', text)
        return text

    # ----------------------------
    # Step 5: Detect & split sections
    # ----------------------------
    def split_sections(self, text: str) -> str:
        for keyword in self.section_keywords:
            pattern = rf'\b({keyword})\b'
            text = re.sub(pattern, r'\n\n\1\n', text)
        return text

    # ----------------------------
    # Step 6: Normalize dates
    # ----------------------------
    def normalize_dates(self, text: str) -> str:
        # Fix common OCR year issues like 2074 -> 2014
        text = re.sub(r'\b(20)(7\d)\b', r'20\2', text)

        # Normalize "to CURRENT"
        text = re.sub(r'\bto CURRENT\b', '- Present', text, flags=re.IGNORECASE)

        # Normalize date ranges
        text = re.sub(
            r'(\d{2}/\d{4})\s*(to|-)\s*(\d{2}/\d{4}|Present)',
            r'\1 - \3',
            text
        )
        return text

    # ----------------------------
    # Step 7: Final formatting
    # ----------------------------
    def finalize(self, text: str) -> str:
        # Ensure spacing between sections
        text = re.sub(r'\n{3,}', '\n\n', text)
        return text.strip()

    # ----------------------------
    # Full pipeline
    # ----------------------------
    def clean(self, raw_text: str) -> str:
        text = raw_text

        text = self.remove_artifacts(text)
        text = self.clean_symbols(text)
        text = self.normalize_dates(text)
        text = self.normalize_whitespace(text)
        text = self.normalize_bullets(text)
        text = self.split_sections(text)
        text = self.finalize(text)

        return text

## 2. Define Agent State and Tools

In [ ]:
import pymupdf4llm

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    raw_text: str
    cleaned_text: str
    sections: Dict[str, str]
    text_insights: Dict[str, object]
    scores: Dict[str, object]
    report_html: str
    chatbot_summary: str
    errors: List[str]

@tool
def add(a: int, b: int):
    """This is an addiction function that adds 2 numbers together"""
    return a + b

@tool
def mutiply(a: int, b: int):
    """This is an exponetial function that mutiply 2 numbers together"""
    return a * b

@tool
def extract_and_clean_text(path):
  text = pymupdf4llm.to_text(path)
  cleaner = CVTextCleaner()
  text_clean = cleaner.clean(text)
  return text_clean


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
tools = [add, mutiply]

## 3. Define model

In [ ]:
from langchain_huggingface import HuggingFacePipeline

model_name = "Qwen/Qwen2-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_llm = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

pipe = pipeline("text-generation", model=model_llm, tokenizer=tokenizer, max_new_tokens=256)
hf_pipe = HuggingFacePipeline(pipeline=pipe)

# Sửa lỗi: Truyền hf_pipe vào tham số llm
model = ChatHuggingFace(llm=hf_pipe)
model = model.bind_tools(tools)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
def model_call(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(
        content="You are my AI assistant, please answer my query to the best of your ability"
    )
    response = model.invoke([system_prompt] + state["messages"])
    return {
        "messages": [response]
    }



In [ ]:
import os
from typing import Dict
from state import AgentState


def get_tavily_client():
    from tavily import TavilyClient

    api_key = os.getenv("TAVILY_API_KEY", "")
    return TavilyClient(api_key=api_key)


def _build_query(text_insights: Dict) -> str:
    skills = text_insights.get("skills", [])
    target_role = text_insights.get("target_role", "")
    skill_text = " ".join(skills[:5]) if isinstance(skills, list) else ""
    parts = [target_role, skill_text, "resume evaluation best practices"]
    return " ".join([part for part in parts if part]).strip()


def rag_enricher(state: AgentState) -> AgentState:
    errors = list(state.get("errors", []))
    api_key = os.getenv("TAVILY_API_KEY", "")

    if not api_key:
        return {"rag_context": {}, "errors": errors}

    text_insights = state.get("text_insights", {})
    query = _build_query(text_insights)
    if not query:
        return {"rag_context": {}, "errors": errors}

    try:
        client = get_tavily_client()
        results = client.search(query, max_results=5)
        return {"rag_context": results, "errors": errors}
    except Exception as exc:
        errors.append(f"rag_enricher:{exc}")
        return {"rag_context": {}, "errors": errors}


In [ ]:
import shutil

# Assuming 'path' variable contains the path to the folder to be zipped
# For example, path = '/root/.cache/kagglehub/datasets/hadikp/resume-data-pdf/versions/1'
# Define the output zip file name and path
output_filename = '/content/zipped_folder'
path="/content/CV_Review_System"
# Create the zip archive
shutil.make_archive(output_filename, 'zip', path)

print(f"Thư mục đã được nén thành: {output_filename}.zip")

Thư mục đã được nén thành: /content/zipped_folder.zip


In [ ]:
def should_continue(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

## 4. Define Graph

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("our_agent", model_call)
tool_node = ToolNode(tools=tools)
graph.add_node("tools", tool_node)
graph.set_entry_point("our_agent")
graph.add_conditional_edges(
    "our_agent", should_continue,
    {
        "continue": "tools",
        "end": END
    }
)
graph.add_edge("tools", "our_agent")
app = graph.compile()

In [ ]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

In [ ]:
from langchain_core.messages import HumanMessage

# Sửa lỗi: Chuyển đổi input sang HumanMessage
inputs = {"messages": [HumanMessage(content="Add 297329 + 37 and tell me a joke")]}
print_stream(app.stream(inputs, stream_mode="values"))

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


================================ Human Message =================================

Add 297329 + 37 and tell me a joke
================================== Ai Message ==================================

<|im_start|>system
You are my AI assistant, please answer my query to the best of your ability<|im_end|>
<|im_start|>user
Add 297329 + 37 and tell me a joke<|im_end|>
<|im_start|>assistant
Sure! Let's do it step by step:

1. Start with 297329: 
   - Add 297329: 306468

2. Now add 37:
   - Add 37: 300057

Now let's combine these two numbers together:

306468 + 300057 = 606525

So, adding 297329 + 37 gives us a total of 606525.

As for a joke, here goes:

Why don't scientists trust atoms?

Because they make up everything!

What does that mean for science? It means that atoms can be made out of anything!


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hadikp/resume-data-pdf")

print("Path to dataset files:", path)

100%|██████████| 1.99G/1.99G [00:20<00:00, 102MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/hadikp/resume-data-pdf/versions/1


In [ ]:
!pip install pymupdf4llm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 18.5 MB/s eta 0:00:00


In [ ]:
ls "/root/.cache/kagglehub/datasets/hadikp/resume-data-pdf/versions/1/Resumes PDF/Data Science"

0.pdf    120.pdf  141.pdf  162.pdf  183.pdf  23.pdf  44.pdf  65.pdf  86.pdf
100.pdf  121.pdf  142.pdf  163.pdf  184.pdf  24.pdf  45.pdf  66.pdf  87.pdf
101.pdf  122.pdf  143.pdf  164.pdf  185.pdf  25.pdf  46.pdf  67.pdf  88.pdf
102.pdf  123.pdf  144.pdf  165.pdf  186.pdf  26.pdf  47.pdf  68.pdf  89.pdf
103.pdf  124.pdf  145.pdf  166.pdf  187.pdf  27.pdf  48.pdf  69.pdf  8.pdf
104.pdf  125.pdf  146.pdf  167.pdf  188.pdf  28.pdf  49.pdf  6.pdf   90.pdf
105.pdf  126.pdf  147.pdf  168.pdf  189.pdf  29.pdf  4.pdf   70.pdf  91.pdf
106.pdf  127.pdf  148.pdf  169.pdf  18.pdf   2.pdf   50.pdf  71.pdf  92.pdf
107.pdf  128.pdf  149.pdf  16.pdf   190.pdf  30.pdf  51.pdf  72.pdf  93.pdf
108.pdf  129.pdf  14.pdf   170.pdf  191.pdf  31.pdf  52.pdf  73.pdf  94.pdf
109.pdf  12.pdf   150.pdf  171.pdf  192.pdf  32.pdf  53.pdf  74.pdf  95.pdf
10.pdf   130.pdf  151.pdf  172.pdf  193.pdf  33.pdf  54.pdf  75.pdf  96.pdf
110.pdf  131.pdf  152.pdf  173.pdf  194.pdf  34.pdf  55.pdf  76.pdf  97.pdf
111.pdf  132.

In [ ]:
import pymupdf4llm
js = pymupdf4llm.to_text("/root/.cache/kagglehub/datasets/hadikp/resume-data-pdf/versions/1/Resumes PDF/Data Science/1.pdf")
cleaner = CVTextCleaner()
text_clean = cleaner.clean(js)
text_clean

=== Document parser messages ===
                                                            Using Tesseract for OCR processing.
OCR on page.number=0/1.



"CLAI RE \n\nSUMMARY\n —_ Forward\n- thinking Bioinformatics and Data Science Engineer with the related experience and dedication @ to make a difference to research and clinical outcomes. Proven skills in evaluating bioinformatics data resumesample@example.com with software\n- based, statistical and data mining approaches. \\ Excellent documentation, reporting and presentation skills. Seeking to offer knowledge and abilities to (555) 432\n- 1000 a dynamic, growth\n- oriented company. 9 100 Montgomery St. 10th Floor \n\nSKILLS\n \n\nEXPERIENCE\n + Over 7 years of expert level Konica Minolta\n- Bioinformatics and Data Science Engineer experience in R\n- Programming Seattle, WA 72/2016\n- Current + 4 years experience working ina\n- Primary responsibilities include demonstrating ORIEN Informatics portal and collect feedback and Linux Environment with bash feature requirements from researchers including biologists, clinical trial designers and clinicians scripting skills : Active contributo

In [ ]:
js

'‘%2 (555) 432-1000 4 resumesample@example.com \n\n==> picture [223 x 13] <==\n\n----- Start of picture text -----\n@ Montgomery Street, San Francisco, CA 94105\n----- End of picture text -----\n\nOBJECTIVE © Data Science and Engineering Leader looking for a senior leadership position with the responsibilities to drive organizational KPls through advance data analytics and use of artificial intelligence to guide development of modern software products and platforms.\n\nEXECUTIVE SUMMARY © Asa Principal/Director of Data Science and Engineering, with 16+ years of professional experience and 7+ in management, | am proficient in Leading Strategic Product Thinking effort, Data Driven Decisioning, SDLC., Planning, Cross Functional and Cross Team Coordination, Team Development, Tactical Reorganization, Consensus Building and Operations Management \n\nAREAS OF EXPERTISE © \n\n« Technology Leadership and Innovation\n\n« Product Planning and OKR Definition\n\n* Leading MVP Definition and Develop

In [ ]:
inputs = {"messages": [HumanMessage(content=f"tóm gọn lại nội dung này cho tôi: {js}")]}
print_stream(app.stream(inputs, stream_mode="values"))

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


================================ Human Message =================================

tóm gọn lại nội dung này cho tôi: ‘%2 (555) 432-1000 4 resumesample@example.com 

==> picture [223 x 13] <==

----- Start of picture text -----
@ Montgomery Street, San Francisco, CA 94105
----- End of picture text -----

OBJECTIVE © Data Science and Engineering Leader looking for a senior leadership position with the responsibilities to drive organizational KPls through advance data analytics and use of artificial intelligence to guide development of modern software products and platforms.

EXECUTIVE SUMMARY © Asa Principal/Director of Data Science and Engineering, with 16+ years of professional experience and 7+ in management, | am proficient in Leading Strategic Product Thinking effort, Data Driven Decisioning, SDLC., Planning, Cross Functional and Cross Team Coordination, Team Development, Tactical Reorganization, Consensus Building and Operations Management 

AREAS OF EXPERTISE © 

« Technology Leade

In [ ]:
!pip install weasyprint

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 64.6 MB/s eta 0:00:00
  Attempting uninstall: tinycss2
    Found existing installation: tinycss2 1.4.0
    Uninstalling tinycss2-1.4.0:
      Successfully uninstalled tinycss2-1.4.0


In [ ]:
from weasyprint import HTML

# 1. Đoạn mã HTML & CSS đã tối ưu
html_content = """
<!DOCTYPE html>
<html lang="vi">
<head>
    <meta charset="UTF-8">
    <title>Báo Cáo Phân Tích Resume</title>
    <style>
        @import url('https://googleapis.com');

        @page {
            size: A4;
            margin: 15mm;
            @bottom-right {
                content: "Trang " counter(page) " / " counter(pages);
                font-family: 'Roboto', sans-serif;
                font-size: 9px;
                color: #7f8c8d;
            }
            @bottom-left {
                content: "DigiSource Resume Scorer";
                font-family: 'Roboto', sans-serif;
                font-size: 9px;
                color: #7f8c8d;
            }
        }

        /* Gọi trực tiếp font Arial có sẵn trong máy */
        body {
            font-family: 'Arial', 'Times New Roman', sans-serif;
            font-size: 11px;
            line-height: 1.4;
            color: #2c3e50;
            background: #fff;
            margin: 0;
            padding: 0;
        }

        @page {
            size: A4;
            margin: 15mm;
            @bottom-right {
                content: "Trang " counter(page) " / " counter(pages);
                font-family: 'Arial', sans-serif;
                font-size: 9px;
                color: #7f8c8d;
            }
            @bottom-left {
                content: "DigiSource Resume Scorer";
                font-family: 'Arial', sans-serif;
                font-size: 9px;
                color: #7f8c8d;
            }
        }

        /* Chống vỡ khối dữ liệu khi ngắt trang */
        h1, h2, h3, .phase-card, .box-grid, .footer-analysis {
            page-break-inside: avoid;
        }

        /* HEADER */
        .header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            border-bottom: 2.5px solid #0052cc;
            padding-bottom: 10px;
            margin-bottom: 15px;
        }
        .header-title h1 {
            color: #0052cc;
            font-size: 20px;
            font-weight: 700;
            margin: 0 0 3px 0;
            text-transform: uppercase;
        }
        .header-title p {
            color: #7f8c8d;
            font-size: 11px;
            margin: 0;
            font-weight: 500;
        }
        .header-meta {
            text-align: right;
            font-size: 10px;
            color: #7f8c8d;
        }

        /* TỔNG QUAN THÔNG TIN & ĐIỂM SỐ */
        .summary-section {
            display: flex;
            gap: 15px;
            margin-bottom: 15px;
        }
        .info-box {
            flex: 2;
            background: #f8f9fa;
            border-left: 4px solid #0052cc;
            padding: 12px;
            border-radius: 4px;
        }
        .info-box h2 {
            font-size: 13px;
            color: #0052cc;
            margin: 0 0 8px 0;
            text-transform: uppercase;
        }
        .info-grid {
            display: grid;
            grid-template-columns: 80px auto;
            row-gap: 5px;
            font-size: 11px;
        }
        .info-label {
            font-weight: 700;
            color: #7f8c8d;
        }
        .info-value {
            color: #2c3e50;
            font-weight: 500;
        }

        .score-box {
            flex: 1;
            background: #ebf5ff;
            border: 1px solid #cce0ff;
            border-radius: 4px;
            text-align: center;
            padding: 10px;
            display: flex;
            flex-direction: column;
            justify-content: center;
        }
        .score-box .score-title {
            font-size: 11px;
            font-weight: 700;
            color: #0052cc;
            text-transform: uppercase;
        }
        .score-box .score-value {
            font-size: 32px;
            font-weight: 700;
            color: #0052cc;
            margin: 3px 0;
        }
        .score-box .score-badge {
            display: inline-block;
            background: #27ae60;
            color: white;
            padding: 2px 8px;
            border-radius: 10px;
            font-size: 10px;
            font-weight: 700;
            width: fit-content;
            margin: 0 auto;
        }

        /* CHI TIẾT ĐIỂM SỐ THEO CÁC GIAI ĐOẠN */
        .section-title {
            font-size: 13px;
            color: #0052cc;
            border-bottom: 1px solid #e0e0e0;
            padding-bottom: 4px;
            margin: 15px 0 10px 0;
            text-transform: uppercase;
            font-weight: 700;
        }
        .phase-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 12px;
            margin-bottom: 12px;
        }
        .phase-card {
            background: #ffffff;
            border: 1px solid #e0e0e0;
            border-radius: 4px;
            padding: 10px;
        }
        .phase-header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            border-bottom: 1px solid #f0f0f0;
            padding-bottom: 5px;
            margin-bottom: 8px;
        }
        .phase-title {
            font-size: 11px;
            font-weight: 700;
            color: #2c3e50;
        }
        .phase-score {
            font-weight: 700;
            color: #0052cc;
            font-size: 11px;
        }
        .sub-criteria {
            margin-bottom: 8px;
        }
        .sub-criteria:last-child {
            margin-bottom: 0;
        }
        .sub-title {
            display: flex;
            justify-content: space-between;
            font-weight: 500;
            color: #34495e;
            margin-bottom: 2px;
            font-size: 10px;
        }
        .sub-score {
            color: #27ae60;
            font-weight: 700;
        }
        .sub-detail {
            color: #7f8c8d;
            font-size: 9.5px;
            line-height: 1.3;
            padding-left: 8px;
            border-left: 2px solid #e0e0e0;
        }

        /* ĐIỂM MẠNH & CẢI THIỆN */
        .box-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 12px;
            margin-bottom: 12px;
        }
        .pro-box {
            background: #e8f5e9;
            border: 1px solid #c8e6c9;
            border-radius: 4px;
            padding: 10px;
        }
        .pro-box h3 {
            color: #2e7d32;
            font-size: 11px;
            margin: 0 0 6px 0;
            font-weight: 700;
        }
        .con-box {
            background: #fff3e0;
            border: 1px solid #ffe0b2;
            border-radius: 4px;
            padding: 10px;
        }
        .con-box h3 {
            color: #e65100;
            font-size: 11px;
            margin: 0 0 6px 0;
            font-weight: 700;
        }
        .box-grid ul {
            margin: 0;
            padding-left: 15px;
            font-size: 10px;
            color: #333;
        }
        .box-grid li {
            margin-bottom: 4px;
        }

        /* GỢI Ý CẢI THIỆN ƯU TIÊN */
        .priority-grid {
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 10px;
            margin-bottom: 15px;
        }
        .priority-item {
            background: #fdf2e9;
            border-top: 3px solid #e67e22;
            padding: 8px;
            border-radius: 3px;
            font-size: 9.5px;
        }
        .priority-item strong {
            display: block;
            color: #e67e22;
            margin-bottom: 3px;
            font-size: 10px;
        }

        /* GỢI Ý CHUNG */
        .general-tips {
            background: #f4f6f7;
            padding: 10px;
            border-radius: 4px;
            margin-bottom: 15px;
        }
        .general-tips h3 {
            font-size: 11px;
            color: #2c3e50;
            margin: 0 0 6px 0;
            font-weight: 700;
        }
        .general-tips ol {
            margin: 0;
            padding-left: 15px;
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 10px;
            font-size: 10px;
        }

        /* TIÊU CHUẨN VÀ TỪ KHÓA */
        .footer-analysis {
            background: #f8f9fa;
            border: 1px solid #e0e0e0;
            border-radius: 4px;
            padding: 10px;
            margin-bottom: 15px;
        }
        .footer-analysis h3 {
            font-size: 11px;
            color: #2c3e50;
            margin: 0 0 4px 0;
            font-weight: 700;
        }
        .footer-analysis p {
            margin: 0 0 8px 0;
            font-size: 10px;
            color: #555;
        }
        .tech-tags {
            display: flex;
            flex-wrap: wrap;
            gap: 4px;
        }
        .tech-tag {
            background: #e9ecef;
            color: #495057;
            padding: 2px 6px;
            border-radius: 3px;
            font-size: 9px;
            font-weight: 500;
        }

        /* FOOTER BRAND */
        .brand-footer {
            text-align: center;
            font-size: 9px;
            color: #95a5a6;
            margin-top: 10px;
            border-top: 1px solid #eee;
            padding-top: 8px;
        }
    </style>
</head>
<body>
    <div class="container">
        <!-- HEADER -->
        <div class="header">
            <div class="header-title">
                <h1>BÁO CÁO PHÂN TÍCH RESUME</h1>
                <p>Hệ thống đánh giá thông minh DigiSource Resume Scorer</p>
            </div>
            <div class="header-meta">
                <div>Ngày tạo: 11/11/2025</div>
                <div>ID: #DS-2025-082</div>
            </div>
        </div>

        <!-- THÔNG TIN VÀ ĐIỂM SỐ -->
        <div class="summary-section">
            <div class="info-box">
                <h2>Thông tin ứng viên</h2>
                <div class="info-grid">
                    <span class="info-label">Tên:</span>
                    <span class="info-value">Nguyen Thanh Duy Tan</span>

                    <span class="info-label">Cấp độ:</span>
                    <span class="info-value">Intern / Fresher</span>

                    <span class="info-label">Ngành nghề:</span>
                    <span class="info-value">Công nghệ thông tin (IT)</span>
                </div>
            </div>
            <div class="score-box">
                <div class="score-title">Tổng điểm</div>
                <div class="score-value">82<span style="font-size:16px; color:#7f8c8d;">/100</span></div>
                <div class="score-badge">TỐT</div>
            </div>
        </div>

        <!-- CHI TIẾT ĐIỂM SỐ -->
        <div class="section-title">Chi tiết điểm số theo giai đoạn</div>

        <div class="phase-grid">
            <!-- PHASE 2 -->
            <div class="phase-card">
                <div class="phase-header">
                    <span class="phase-title">PHASE 2 - CORE FOUNDATION</span>
                    <span class="phase-score">60 điểm</span>
                </div>
                <div class="sub-criteria">
                    <div class="sub-title">Format & ATS Compliance <span class="sub-score">17/20</span></div>
                    <div class="sub-detail">Định dạng file PDF (Word export). Kích thước &lt;1MB. Tên file "NguyenThanhDuyTan_CV_Fullstack.pdf (1).pdf" cần chuẩn hóa lại như "NguyenThanhDuyTan_Fullstack_Fresher.pdf".</div>
                </div>
                <div class="sub-criteria">
                    <div class="sub-title">Professional Foundation <span class="sub-score">13/20</span></div>
                    <div class="sub-detail">Thông tin họ tên và email đầy đủ. Số điện thoại "0339900276" bị thiếu mã quốc gia.</div>
                </div>
                <div class="sub-criteria">
                    <div class="sub-title">Content Quality <span class="sub-score">18/20</span></div>
                    <div class="sub-detail">Có một vài lỗi nhỏ hoặc cách diễn đạt chưa tự nhiên. Cần tăng độ đa dạng của động từ hành động và giảm từ ngữ filler chung chung.</div>
                </div>
            </div>

            <!-- PHASE 3 -->
            <div class="phase-card">
                <div class="phase-header">
                    <span class="phase-title">PHASE 3 - SPECIALIZED ASSESSMENT</span>
                    <span class="phase-score">30 điểm</span>
                </div>
                <div class="sub-criteria">
                    <div class="sub-title">Experience Assessment <span class="sub-score">15/15</span></div>
                    <div class="sub-detail">Cấp độ Fresher nên chưa có sự thăng tiến rõ ràng. Mô tả dự án chi tiết nhưng thiếu định lượng kết quả và động từ mạnh.</div>
                </div>
                <div class="sub-criteria">
                    <div class="sub-title">Technical Proof <span class="sub-score">6/8</span></div>
                    <div class="sub-detail">Có link GitHub chung và link demo cho từng dự án. Tech stack được liệt kê rõ ràng nhưng chưa thể hiện mức độ thành thạo.</div>
                </div>
                <div class="sub-criteria">
                    <div class="sub-title">Portfolio/Projects <span class="sub-score">5/12</span></div>
                    <div class="sub-detail">Dự án đa dạng (LMS, Music, Quiz) có key features. Không có chứng chỉ (0/5) và không có điểm cộng kiến thức ngành.</div>
                </div>
            </div>
        </div>

        <!-- PHASE 4 -->
        <div class="phase-card" style="margin-bottom: 15px;">
            <div class="phase-header">
                <span class="phase-title">PHASE 4 - BONUS FACTORS</span>
                <span class="phase-score">8/10 điểm</span>
            </div>
            <div class="sub-criteria">
                <div class="sub-title">Differentiating Factors <span class="sub-score">8/10</span></div>
                <div class="sub-detail">Có bằng chứng về Leadership thông qua việc quản lý các dự án fullstack. Tuy nhiên chưa có kinh nghiệm quốc tế và giải thưởng.</div>
            </div>
        </div>

        <!-- ĐIỂM MẠNH VÀ CẢI THIỆN -->
        <div class="box-grid">
            <div class="pro-box">
                <h3>👍 Điểm mạnh</h3>
                <ul>
                    <li>CV có cấu trúc rõ ràng, bố cục trực quan và rất dễ đọc.</li>
                    <li>Phần dự án trình bày chi tiết, áp dụng kiến thức vào thực tế tốt.</li>
                    <li>Sử dụng đa dạng các công nghệ và framework hiện đại.</li>
                    <li>Thông tin liên hệ đầy đủ và chuyên nghiệp.</li>
                </ul>
            </div>
            <div class="con-box">
                <h3>🔧 Cần cải thiện (Gợi ý chung)</h3>
                <ul>
                    <li>Bổ sung phần "Professional Summary" (Tóm tắt mục tiêu & điểm mạnh).</li>
                    <li>Định lượng hóa các kết quả cụ thể trong phần mô tả dự án.</li>
                    <li>Phân loại kỹ năng rõ hơn (Frontend, Backend, Database) + mức thành thạo.</li>
                    <li>Bổ sung các chứng chỉ liên quan đến IT để tăng tính cạnh tranh.</li>
                </ul>
            </div>
        </div>

        <!-- ƯU TIÊN HÀNH ĐỘNG -->
        <div class="section-title">Gợi ý cải thiện ưu tiên (Priority Actions)</div>
        <div class="priority-grid">
            <div class="priority-item">
                <strong>1. Dự án lớn hơn</strong>
                Bổ sung kinh nghiệm thực tế hoặc các dự án cá nhân có quy mô lớn hơn, kiến trúc phức tạp hơn.
            </div>
            <div class="priority-item">
                <strong>2. Số liệu hóa</strong>
                Định lượng hóa kết quả trong dự án (ví dụ: giảm thời gian tải trang, tăng tỉ lệ người dùng).
            </div>
            <div class="priority-item">
                <strong>3. Phân loại Kỹ năng</strong>
                Cải thiện phần kỹ năng bằng cách phân loại rõ ràng (Frontend, Backend, Database) và mức độ.
            </div>
            <div class="priority-item">
                <strong>4. Tìm kiếm Chứng chỉ</strong>
                Săn tìm các chứng chỉ uy tín liên quan đến công nghệ mục tiêu để gia tăng cạnh tranh.
            </div>
        </div>

        <!-- GỢI Ý CHUNG -->
        <div class="general-tips">
            <h3>💡 Lời khuyên phát triển nghề nghiệp</h3>
            <ol>
                <li>Tìm kiếm cơ hội thực tập hoặc Fresher để va chạm và tích lũy kinh nghiệm thực tế.</li>
                <li>Tiếp tục phát triển các dự án cá nhân với các công nghệ mới và bài toán khó hơn.</li>
                <li>Tham gia các khóa học online hoặc chương trình đào tạo để lấy chứng chỉ.</li>
                <li>Xây dựng mạng lưới quan hệ trong ngành IT thông qua LinkedIn hoặc sự kiện networking.</li>
            </ol>
        </div>

        <!-- TIÊU CHUẨN VÀ TỪ KHÓA -->
        <div class="footer-analysis">
            <h3>🎯 Tiêu chuẩn ngành (Fresher/Intern IT)</h3>
            <p>Nhà tuyển dụng tìm kiếm ứng viên có nền tảng kiến thức vững chắc, khả năng tự học tốt, có dự án cá nhân thể hiện sự chủ động và đam mê. CV cần làm nổi bật kỹ năng kỹ thuật, công nghệ đã sử dụng và cách ứng dụng giải quyết vấn đề.</p>

            <h3>🏷️ Từ khóa ngành xuất hiện trong CV</h3>
            <div class="tech-tags">
                <span class="tech-tag">HTML5</span><span class="tech-tag">CSS3</span>
                <span class="tech-tag">JavaScript</span><span class="tech-tag">ReactJS</span>
                <span class="tech-tag">Redux Toolkit</span><span class="tech-tag">TypeScript</span>
                <span class="tech-tag">Node.js</span><span class="tech-tag">Express.js</span>
                <span class="tech-tag">FastAPI</span><span class="tech-tag">MongoDB</span>
                <span class="tech-tag">PostgreSQL</span>
            </div>
        </div>

        <!-- FOOTER -->
        <div class="brand-footer">
            Được tạo bởi DigiSource Resume Scorer • <a href="https://digisource.vn" style="color: #95a5a6; text-decoration: none;">digisource.vn</a>
        </div>
    </div>
</body>
</html>
"""

# 2. Xuất dữ liệu ra file PDF
HTML(string=html_content).write_pdf("bao_cao_resume_hoan_hao.pdf")
print("Đã xuất file báo cáo PDF thành công!")


ERROR:weasyprint:Failed to load stylesheet at https://googleapis.com : HTTPError: HTTP Error 404: Not Found
DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.002s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.004s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
INFO:fontTools.subset:kern dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INF

Đã xuất file báo cáo PDF thành công!
